# Notebook 02 – EDA after Cohort Extraction


--------

Cohort Population Extracted from 01 notebook

**Main Objectives**

+ Verify that data missingness reflects real-world medical patterns
+ Verify assumptions based off feature distributions and sample sizes
+ Baseline metrics are established and able to be used


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import missingno as msno

from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.metrics import roc_auc_score

In [ ]:
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi' : 300})


np.random.seed(617)
n_patients = 1000
stay_ids = np.arange(10000, 10000 + n_patients)

cohort = pd.DataFrame(
    {
        'stay_id' : stay_ids,
        'age' : np.random.normal(65, 15, n_patients).clip(18,100),
        'gender' : np.random.choice(['M', 'F'], n_patients),
        'race' : np.random.choice(['White', 'Black', 'Hispanic', 'Asian', 'Other', 'Unknown'], n_patients),
        'insurance' : np.random.choice(['Medicare', 'Medicaid', 'Private', 'Other'], n_patients),
        'mortality_24h' : np.random.binomial(1, 0.15, n_patients)
    }
)

ts_records = []

for sid, outcome in zip(cohort['stay_id'], cohort['mortality_24h']):
    for t in range(24):
        hr = np.random.normal(80 + (t*0.5 if outcome else 0), 15)
        map_ = np.random.normal(85 - (t*0.5 if outcome else 0), 10)
        lactate = np.random.exponential(1.5 + (t*0.1 if outcome else 0))
        gcs = np.random.norma(14 - (t*0.2 if outcome else 0), 2).clip(3, 15)
        
        if np.random.rand() < 0.2: hr - np.nan
        if np.random.rand() < 0.6: lactate - np.nan
        
        ts_records.append([sid, t, hr, map_, lactate, gcs])
ts = pd.DataFrame(ts_records, columns=['stay_id', 'time_step', 'heart_rate', 'mbp', 'lactate', 'gcs_total'])
        